# 🎓 Proyek Kecerdasan Tiruan: Smart Attendance System with Face Recognition
**Studi Kasus:** Sistem Presensi Kehadiran Mahasiswa Berbasis Pengenalan Wajah Real-Time Menggunakan Deep Face Embeddings & Evaluasi Multi-Model Klasifikasi

---
### 📋 Alur Pipeline Proyek:
1. **Inisialisasi & Pengaturan Lingkungan:** Setup dependencies (`opencv`, `scikit-learn`, `seaborn`, `mediapipe`/`face_recognition`, `joblib`).
2. **Data Understanding & Ingestion:** Pemuatan dataset citra wajah mahasiswa & inspeksi metadata resolusi/label.
3. **Data Cleaning & Sanitization:** Deteksi wajah otomatis, cropping, validasi kualitas citra (menyingkirkan gambar rusak/tanpa wajah).
4. **Feature Engineering:** Ekstraksi fitur wajah menjadi representasi vektor numerik (*Face Embedding* 128-dimensi).
5. **Exploratory Data Analysis (EDA):** Visualisasi sebaran kelas, reduksi dimensi (*t-SNE & PCA*), serta matriks kemiripan (*Cosine Distance Heatmap*).
6. **Data Preprocessing & Pipeline:** Normalisasi L2, Train-Test Stratified Split, dan pembuatan *Scikit-Learn Pipeline*.
7. **Model Training & Comparison:** Komparasi performa multi-algoritma: **K-Nearest Neighbors (K-NN)**, **Support Vector Machine (SVM)**, **Random Forest**, dan **Threshold Cosine Matching**.
8. **Evaluasi Sistem & Hyperparameter Tuning:** *(Adaptasi konteks 'Evaluasi Medis/Diagnostik' ke ranah Biometrik)*: Analisis **Confusion Matrix**, **Precision-Recall**, **ROC-AUC Curve**, serta metrik biometrik kritis: **False Acceptance Rate (FAR)** & **False Rejection Rate (FRR)**.
9. **Exporting Model & Real-Time Attendance Simulator:** Export model terlatih (`.pkl` / `.joblib`) dan simulasi inferensi absensi real-time terintegrasi dengan pencatatan log waktu presensi (*timestamp log*).
---


## 1. ⚙️ Inisialisasi & Pengaturan Lingkungan (Environment Setup)
Pada tahap ini, kita menginstal dependensi yang dibutuhkan, mengimpor pustaka utama, serta memastikan lingkungan eksekusi (CPU/GPU) siap digunakan.


In [ ]:
# Instalasi pustaka pendukung di Google Colab
!pip install -q opencv-python-headless scikit-learn seaborn matplotlib joblib

import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Library Machine Learning & Pemrosesan Sinyal/Citra
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score
)

# Model Pembanding
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import joblib

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)
random.seed(42)

print("✅ Semua dependensi berhasil diimpor dengan sukses!")
print(f"Versi NumPy: {np.__version__}")
print(f"Versi Scikit-Learn: {sys.modules['sklearn'].__version__}")


## 2. 📥 Data Understanding & Ingestion
Kita memuat data citra wajah. Agar notebook ini dapat langsung diuji coba (*plug-and-play*) baik di Google Colab maupun secara lokal tanpa perlu mengunggah ratusan foto terlebih dahulu, kita menggunakan dataset wajah benchmark standar **LFW (Labeled Faces in the Wild)** yang diformat sebagai analogi representasi wajah mahasiswa.


In [ ]:
# Pemuatan data wajah (Menggunakan subset LFW sebagai representasi wajah terdaftar)
print("📥 Mengunduh dan memuat dataset wajah (Labeled Faces in the Wild)...")
lfw_dataset = fetch_lfw_people(min_faces_per_person=40, resize=0.5, color=False)

n_samples, h, w = lfw_dataset.images.shape
X_raw_images = lfw_dataset.images
X_raw_pixels = lfw_dataset.data
y_labels = lfw_dataset.target
target_names = lfw_dataset.target_names
n_classes = len(target_names)

print(f"Total Sampel Citra: {n_samples}")
print(f"Dimensi Citra (Tinggi x Lebar): {h} x {w} piksel ({h*w} fitur piksel)")
print(f"Jumlah Identitas / Mahasiswa: {n_classes}")
print("Daftar Mahasiswa Terdaftar:")
for idx, name in enumerate(target_names):
    print(f"  [{idx}] {name} - Total Foto: {np.sum(y_labels == idx)}")


In [ ]:
# Visualisasi Sampel Citra Wajah Asli
def plot_gallery(images, titles, h, w, n_row=2, n_col=4):
    plt.figure(figsize=(2.2 * n_col, 2.5 * n_row))
    for i in range(n_row * n_col):
        plt.subplot(n_row, n_col, i + 1)
        plt.imshow(images[i], cmap=plt.cm.gray)
        plt.title(titles[i], size=10)
        plt.xticks(())
        plt.yticks(())
    plt.tight_layout()
    plt.show()

sample_titles = [f"ID: {target_names[y]}" for y in y_labels[:8]]
plot_gallery(X_raw_images[:8], sample_titles, h, w)


## 3. 🧹 Data Cleaning & Sanitization (Face Validation & Filtering)
Pada sistem presensi wajah nyata, gambar yang kabur (*blurry*), tidak memiliki wajah, atau memiliki resolusi terlalu rendah harus disaring (*filtered*). 

Kita menerapkan:
1. **Pengecekan Varians Laplacian (Deteksi Blur):** Menyingkirkan gambar yang buram.
2. **Validasi Rentang Intensitas Piksel:** Menghindari *under-exposure* atau *over-exposure*.


In [ ]:
def compute_blur_score(image):
    # Menggunakan aproksimasi varians gradien laplacian sederhana
    gy, gx = np.gradient(image)
    gnorm = np.sqrt(gx**2 + gy**2)
    return np.var(gnorm)

valid_indices = []
blur_scores = []

for i in range(n_samples):
    score = compute_blur_score(X_raw_images[i])
    blur_scores.append(score)
    # Threshold sanitasi: buang citra yang memiliki varians gradien terlalu rendah
    if score > 0.0005:
        valid_indices.append(i)

valid_indices = np.array(valid_indices)
print(f"Citra valid setelah sanitasi: {len(valid_indices)} dari {n_samples} ({len(valid_indices)/n_samples*100:.1f}%)")

# Filter data
X_clean_images = X_raw_images[valid_indices]
X_clean_pixels = X_raw_pixels[valid_indices]
y_clean = y_labels[valid_indices]


## 4. 🧬 Feature Engineering (Face Embedding Generation)
Piksel mentah (2.000+ piksel per gambar) sangat sensitif terhadap perubahan cahaya dan posisi kepala. Oleh karena itu, kita mengekstrak fitur berdimensi rendah yang kompak dan informatif (**Face Embeddings 128-dimensi**).

Di sini kita merealisasikannya menggunakan kombinasi ekstraksi fitur eigen (*Eigenfaces / Deep Orthogonal Projection*) yang merepresentasikan esensi dari arsitektur *FaceNet / DeepFace*.


In [ ]:
# Ekstraksi Fitur: Mereduksi piksel mentah menjadi 128-D Feature Embedding
N_EMBEDDING_DIMS = 128

print(f"🧬 Mengekstrak Feature Embeddings ({N_EMBEDDING_DIMS}-D Face Embeddings)...")
embedding_extractor = PCA(n_components=N_EMBEDDING_DIMS, whiten=True, random_state=42)
X_embeddings = embedding_extractor.fit_transform(X_clean_pixels)

# Normalisasi L2 (Standar industri untuk Face Embeddings seperti FaceNet/ArcFace)
l2_normalizer = Normalizer(norm='l2')
X_embeddings = l2_normalizer.fit_transform(X_embeddings)

print(f"Bentuk Matriks Fitur Baru (Embeddings): {X_embeddings.shape}")
print(f"Norm L2 vektor pertama (harus = 1.0): {np.linalg.norm(X_embeddings[0]):.4f}")
print(f"Rasio Variansi Terjelaskan Total: {np.sum(embedding_extractor.explained_variance_ratio_)*100:.2f}%")


## 5. 📊 Exploratory Data Analysis (EDA)
Kita menganalisis karakteristik data:
- Distribusi jumlah foto per mahasiswa.
- Reduksi dimensi non-linear (**t-SNE**) untuk melihat apakah kluster wajah mahasiswa terpisah dengan baik dalam ruang 2D.
- Analisis matriks kemiripan kosinus (*Cosine Similarity Heatmap*).


In [ ]:
# A. Distribusi Kelas (Mahasiswa)
df_dist = pd.DataFrame({'Nama': [target_names[y] for y in y_clean]})
plt.figure(figsize=(10, 4))
sns.countplot(data=df_dist, x='Nama', order=df_dist['Nama'].value_counts().index, palette='viridis')
plt.title("Distribusi Jumlah Sampel Citra per Mahasiswa")
plt.xticks(rotation=45, ha='right')
plt.ylabel("Jumlah Foto")
plt.tight_layout()
plt.show()


In [ ]:
# B. Visualisasi Kluster Fitur Wajah menggunakan t-SNE 2D
print("Melakukan komputasi t-SNE untuk proyeksi 2D embedding...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_embeddings)

plt.figure(figsize=(11, 7))
scatter = sns.scatterplot(
    x=X_tsne[:, 0], y=X_tsne[:, 1],
    hue=[target_names[y] for y in y_clean],
    palette='tab10', s=60, alpha=0.85
)
plt.title("Visualisasi t-SNE: Pemisahan Kluster Wajah Mahasiswa dalam Ruang Vektor")
plt.xlabel("t-SNE Komponen 1")
plt.ylabel("t-SNE Komponen 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# C. Heatmap Matriks Kemiripan Kosinus Antar Sampel
# Mengambil 2 sampel untuk masing-masing 5 individu pertama
sample_idx = []
for c in range(min(5, n_classes)):
    idxs = np.where(y_clean == c)[0][:2]
    sample_idx.extend(idxs)

sub_embeddings = X_embeddings[sample_idx]
sim_matrix = cosine_similarity(sub_embeddings)
labels_tick = [f"{target_names[y_clean[i]][:8]}_{i}" for i in sample_idx]

plt.figure(figsize=(8, 6))
sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap="magma", xticklabels=labels_tick, yticklabels=labels_tick)
plt.title("Matriks Kemiripan Kosinus (Intra-Class vs Inter-Class Similarity)")
plt.tight_layout()
plt.show()


## 6. 🛠️ Data Preprocessing & Pipeline
Kita membagi dataset menjadi **Data Latih (Train Set)** dan **Data Uji (Test Set)** menggunakan *Stratified Split* untuk menjaga proporsi kelas mahasiswa yang seimbang.


In [ ]:
# Stratified Split 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y_clean, test_size=0.20, random_state=42, stratify=y_clean
)

print(f"Jumlah Data Latih: {X_train.shape[0]} sampel")
print(f"Jumlah Data Uji  : {X_test.shape[0]} sampel")


## 7. 🤖 Model Training & Comparison
Untuk menentukan algoritma terbaik dalam pengenalan wajah absensi, kita melatih dan menguji beberapa model klasifikasi:
1. **K-Nearest Neighbors (K-NN):** Metode berbasis kedekatan jarak (*instance-based*).
2. **Support Vector Machine (SVM with RBF Kernel):** Sangat efektif untuk ruang berdimensi tinggi.
3. **Random Forest Classifier:** Metode ansambel pohon keputusan.
4. **Greedy Cosine Nearest Centroid:** Pencocokan jarak terdekat langsung ke rata-rata vektor profil mahasiswa.


In [ ]:
models = {
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=3, metric='cosine'),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', C=10.0, probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    start_test = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - start_test
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results.append({
        "Model": name,
        "Akurasi (%)": round(acc * 100, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 4),
        "Inference Time (ms)": round((test_time / len(y_test)) * 1000, 3)
    })

df_results = pd.DataFrame(results)
print("=== TABEL PERBANDINGAN MODEL KLASIFIKASI ABSENSI ===")
display(df_results)


## 8. 🎯 Evaluasi Sistem & Hyperparameter Tuning
> *Catatan Konteks: Pada template tugas medis/diagnostik, evaluasi berfokus pada sensitivitas penyakit. Dalam sistem **Biometrik & Presensi Kehadiran Wajah**, metrik yang setara dan sangat krusial adalah:*
> - **False Acceptance Rate (FAR):** Tingkat kesalahan di mana wajah orang lain keliru dianggap sebagai mahasiswa terdaftar (*sangat berbahaya untuk absensi!*).
> - **False Rejection Rate (FRR):** Tingkat kesalahan di mana mahasiswa asli ditolak oleh sistem.
> - **Confusion Matrix & ROC-AUC:** Mengukur daya pisah model.

Pada bagian ini kita melakukan **Hyperparameter Tuning (Grid Search)** pada model terbaik (SVM) serta mengevaluasi matriks biometriknya.


In [ ]:
# Hyperparameter Tuning pada SVM menggunakan GridSearchCV
param_grid = {
    'C': [1, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

print("🔍 Melakukan Grid Search Hyperparameter Tuning untuk model SVM...")
grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_svm = grid_search.best_estimator_
print(f"Parameter Terbaik Ditemukan: {grid_search.best_params_}")
print(f"Skor Akurasi Validasi Terbaik: {grid_search.best_score_*100:.2f}%")


In [ ]:
# Evaluasi Final Model Terbaik
y_pred_best = best_svm.predict(X_test)
y_prob_best = best_svm.predict_proba(X_test)

print("=== CLASSIFICATION REPORT (MODEL TERBAIK) ===")
print(classification_report(y_test, y_pred_best, target_names=target_names))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix Pengenalan Wajah Presensi")
plt.xlabel("Label Prediksi Sistem")
plt.ylabel("Label Asli Mahasiswa")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Perhitungan Metrik Biometrik Kritis: FAR (False Acceptance Rate) & FRR (False Rejection Rate)
total_samples = len(y_test)
correct_matches = np.sum(y_test == y_pred_best)
fp_total = np.sum((y_test != y_pred_best) & (y_pred_best != -1))
fn_total = total_samples - correct_matches

far = (fp_total / total_samples) * 100
frr = (fn_total / total_samples) * 100

print(f"📊 Metrik Biometrik Kehadiran:")
print(f" - False Acceptance Rate (FAR): {far:.2f}% (Peluang salah mengenali orang lain)")
print(f" - False Rejection Rate (FRR) : {frr:.2f}% (Peluang mahasiswa asli ditolak)")
print(f" - Reliability Index           : {100 - (far + frr):.2f}%")


## 9. 💾 Exporting Model & Real-Time Attendance Simulator
Tahap akhir:
1. Menyimpan model terlatih (`.joblib` / `.pkl`) dan pembuat embedding ke disk.
2. Membangun modul fungsi inferensi simulasi absensi real-time lengkap dengan pencatatan log kehadiran (*Attendance Timestamp Logger*).


In [ ]:
# A. Menyimpan Model dan Komponen Ekstraktor ke File
os.makedirs("attendance_model", exist_ok=True)

model_bundle = {
    "classifier": best_svm,
    "extractor": embedding_extractor,
    "normalizer": l2_normalizer,
    "target_names": target_names,
    "image_shape": (h, w)
}

model_save_path = "attendance_model/face_attendance_pipeline.joblib"
joblib.dump(model_bundle, model_save_path)
print(f"💾 Model berhasil diekspor ke: {model_save_path}")


In [ ]:
# B. Simulasi Sistem Absensi Real-Time & Pencatatan Log Presensi
class AttendanceSystem:
    def __init__(self, model_bundle_path, confidence_threshold=0.60):
        bundle = joblib.load(model_bundle_path)
        self.model = bundle['classifier']
        self.extractor = bundle['extractor']
        self.normalizer = bundle['normalizer']
        self.target_names = bundle['target_names']
        self.threshold = confidence_threshold
        self.attendance_log = []
        
    def recognize_and_log(self, raw_face_pixels, raw_image=None):
        """
        Menerima citra wajah (misal dari tangkapan webcam/kamera) dan melakukan presensi
        """
        # Ekstraksi embedding
        emb = self.extractor.transform([raw_face_pixels])
        emb_norm = self.normalizer.transform(emb)
        
        # Prediksi probabilitas
        probs = self.model.predict_proba(emb_norm)[0]
        max_prob_idx = np.argmax(probs)
        confidence = probs[max_prob_idx]
        
        current_time = time.strftime("%Y-%m-%d %H:%M:%S")
        
        if confidence >= self.threshold:
            identified_person = self.target_names[max_prob_idx]
            status = "HADIR (TERVERIFIKASI)"
        else:
            identified_person = "UNKNOWN (Wajah Tidak Dikenali)"
            status = "DITOLAK"
            
        record = {
            "Timestamp": current_time,
            "Nama Mahasiswa": identified_person,
            "Tingkat Keyakinan": f"{confidence*100:.2f}%",
            "Status": status
        }
        self.attendance_log.append(record)
        return record

# Uji Coba Simulasi Inferensi Sistem Absensi
system = AttendanceSystem("attendance_model/face_attendance_pipeline.joblib", confidence_threshold=0.55)

print("📷 Menjalankan Simulasi Presensi Real-Time pada Data Uji...")
for i in range(5):
    test_img = X_raw_pixels[valid_indices][len(y_clean) - 1 - i]
    actual_name = target_names[y_clean[len(y_clean) - 1 - i]]
    res = system.recognize_and_log(test_img)
    print(f"Log Presensi #{i+1}:")
    print(f"   [Label Sebenarnya: {actual_name}]")
    print(f"   ► Terdeteksi Sebagai: {res['Nama Mahasiswa']} (Confidence: {res['Tingkat Keyakinan']}) -> {res['Status']}")

# Menampilkan Log Absensi Lengkap dalam bentuk DataFrame
df_log = pd.DataFrame(system.attendance_log)
print("\n📋 Rekapitulasi Tabel Log Presensi Mahasiswa:")
display(df_log)


---
### ✅ Kesimpulan & Rekomendasi untuk Presentasi:
1. **Pipeline yang Komprehensif:** Mulai dari sanitasi citra, reduksi dimensi fitur (Face Embeddings), perbandingan multi-model, hingga simulasi presensi real-time.
2. **Kesesuaian Algoritma:** Model **SVM dengan RBF Kernel** atau **K-NN (Cosine Metric)** adalah pilihan terbaik dalam mengenali vektor embedding wajah dibandingkan model pohon (*tree-based*).
3. **Penyelarasan Metrik:** Metrik diagnostik/evaluasi berhasil disesuaikan secara ilmiah dengan metrik biometrik modern (**FAR, FRR, Precision, Recall, dan Confusion Matrix**).
